# Парсинг данных с сайта на Python

### Цель:

- Запарсить название, год, страна, жанр, краткое описание


In [19]:
# https://myshows.me

In [2]:
# Импорт необходимых библеотек

In [31]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import time
import re

In [2]:
# Получение информации

In [3]:
url = 'https://www.kino-teatr.ru/rating2/movie/ros/film/v100/' # страница со всеми фильмами
page = requests.get(url)

In [22]:
page.status_code

200

In [49]:
result_list = {'name': [], 'description': [], 'country': [], 'year': [], 'genre': []}

In [50]:
base_url = 'https://www.kino-teatr.ru/rating2/movie/ros/film/v100/r'

pagenum = 1
movies_collected = 0
target_count = 250 

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
while movies_collected < target_count:
    url = f'{base_url}{pagenum}/'
    print(f"Страница: {url}")
    
    try:
        page = requests.get(url,headers=headers)
        soup = bs(page.text, 'html.parser')
        movie_cells = soup.find_all('td', class_='simple_table_data')
        
        for cell in movie_cells:
            if movies_collected >= target_count:
                break
                
            link_tag = cell.find('a', href=True)
            
            name = link_tag.get_text(strip=True)
            
            year_text = ''
            
            genre_span = cell.find('span', class_='small silver')
            genre = genre_span.get_text(strip=True) if genre_span else 'Не указан'
            
            movie_url = 'https://www.kino-teatr.ru' + link_tag['href']
            
            try:
                movie_page = requests.get(movie_url, headers=headers)
                movie_soup = bs(movie_page.text, 'html.parser')
                
                description = 'Нет описания'
                desc_div = movie_soup.find('div', {'itemprop': 'description'})
                if desc_div:
                    description = desc_div.get_text(strip=True)
                    
                
                country_param = movie_soup.find('div', class_='info_table_param', string=re.compile(r'Страна', re.I))
                
                country_row = movie_soup.find(text=re.compile(r'Страна', re.I))
                if country_row:
                    parent = country_row.find_parent()
                    if parent:
                        country_data = parent.find_next_sibling()
                        if country_data:
                            country = country_data.get_text(strip=True)
                
                if not year_text:
                    year_param = movie_soup.find('div', class_='info_table_param', string=re.compile(r'Год|Rod', re.I))
                    if year_param:
                        year_data = year_param.find_next_sibling('div', class_='info_table_data')
                        if year_data:
                            year_text = year_data.get_text(strip=True)
                
                result_list['name'].append(name)
                result_list['year'].append(year_text)
                result_list['country'].append(country)
                result_list['genre'].append(genre)
                result_list['description'].append(description)
                
                movies_collected += 1
                print(f"  [{movies_collected}/{target_count}] {name} {year_text}  {genre}")
                
                time.sleep(2.5)  
                
            except Exception as e:
                print(f"  Ошибка при загрузке {movie_url}: {e}")
                continue  
        
        time.sleep(1)
        
    except Exception as e:
        print(f"Ошибка на странице {url}: {e}")
    
    pagenum += 1


Страница: https://www.kino-teatr.ru/rating2/movie/ros/film/v100/r1/


C:\Users\Admin\AppData\Local\Temp\ipykernel_12280\2353762181.py:50: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  country_row = movie_soup.find(text=re.compile(r'Страна', re.I))


  [1/250] Сестрёнка 2019  военный фильм, драма
  [2/250] Шугалей-3 2021  боевик, приключения
  [3/250] Женская логика - 1 2002  детектив, экранизация
  [4/250] Доктор 2022  драма
  [5/250] Жили-были 2017  мелодрама
  [6/250] Дедушка 2016  мелодрама
  [7/250] Некрасивая подружка - 2. Чёрный кот 2020  детектив
  [8/250] Идеальная пара 2014  мелодрама
  [9/250] От печали до радости 2020  мелодрама
  [10/250] Ты есть... 1993  мелодрама, экранизация
  [11/250] В августе 44-го 2000  военный фильм, шпионский фильм, экранизация
  [12/250] Пальма 2021  драма, семейное кино
  [13/250] Призрак 2015  комедия, мистика
  [14/250] Крик тишины 2019  военный фильм, драма, социальная драма, экранизация
  [15/250] Луной был полон сад 2000  драма
  [16/250] Любовь под прикрытием 2010  криминальный фильм, мелодрама, экранизация
  [17/250] Не хочу жениться! 1993  комедия
  [18/250] Монах и бес 2016  комедия, мистика, фэнтези
  [19/250] Двенадцать чудес 2017  мелодрама
  [20/250] Всё о его бывшей 2018  мелод

In [51]:
result_list


{'name': ['Сестрёнка',
  'Шугалей-3',
  'Женская логика - 1',
  'Доктор',
  'Жили-были',
  'Дедушка',
  'Некрасивая подружка - 2. Чёрный кот',
  'Идеальная пара',
  'От печали до радости',
  'Ты есть...',
  'В августе 44-го',
  'Пальма',
  'Призрак',
  'Крик тишины',
  'Луной был полон сад',
  'Любовь под прикрытием',
  'Не хочу жениться!',
  'Монах и бес',
  'Двенадцать чудес',
  'Всё о его бывшей',
  'Классик',
  'Воспитание жестокости у женщин и собак',
  'Ворошиловский стрелок',
  'Время собирать камни',
  'Убийство на Ждановской',
  'Люблю тебя любую',
  'Особенности национальной рыбалки',
  'Кушать подано!',
  'Коля-перекати поле',
  'Грешник',
  'Некрасивая подружка - 9. Страшная, страшная сказка',
  'Некрасивая подружка - 4. Любовный квадрат',
  'Война',
  'Жена по совместительству',
  'Ненормальный',
  'Стрелец неприкаянный',
  'Он – дракон',
  'Менялы',
  'Прощальные гастроли',
  'Майский дождь',
  'Некрасивая подружка - 14. Ищите женщину',
  'Сёстры',
  'Некрасивая подружка 

In [52]:
file_name = '250ruFilm.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)

In [53]:
df.head()

,name,description,country,year,genre
0,Сестрёнка,Экранизация повести Мустая Карима “Радость наш...,Россия,2019,"военный фильм, драма"
1,Шугалей-3,После долгого плена в ливийской тюрьме «Митига...,Россия,2021,"боевик, приключения"
2,Женская логика - 1,По мотивам одноименного романа Виктора Пронина...,Россия,2002,"детектив, экранизация"
3,Доктор,"Основано на реальных событиях, по документальн...",Россия,2022,драма
4,Жили-были,Старинная деревня в русской глубинке. Из жител...,Россия,2017,мелодрама


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         250 non-null    object
 1   description  250 non-null    object
 2   country      250 non-null    object
 3   year         250 non-null    object
 4   genre        250 non-null    object
dtypes: object(5)
memory usage: 9.9+ KB


In [55]:
df.shape

(250, 5)